# DriveMate 공매 가격 모델 비교

GitHub 검토용 코드 전용 노트북입니다. 데이터·모델·키·실행 출력을 포함하지 않습니다. 저장소의 `backend/scripts` 구현을 재사용합니다.

먼저 저장소 코드와 `backend/requirements-ml.txt` 환경을 준비하고, 사용 권한이 있는 온비드 수집 폴더를 별도로 지정하세요. 해당 폴더에는 수집 완료 `manifest.json`과 그 매니페스트가 참조하는 원본 페이지 파일이 있어야 합니다.

공매 낙찰가를 평가하며 일반 소비자 시세를 검증하는 자료가 아닙니다. 미래 차령·주행거리 입력은 시장 조건 고정 시나리오입니다. 이 노트북은 외부 수집 API나 LLM을 호출하지 않습니다. 실행 후 저장된 출력은 GitHub에 올리기 전에 모두 지우세요.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import sys

# 저장소 루트에서 실행하거나 이 경로를 직접 바꾸세요. Colab 예: Path('/content/make_drive_mate')
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
REPO_ROOT = REPO_ROOT.resolve()
# 별도로 준비한 수집 폴더로 변경하세요. 데이터는 노트북에 포함되지 않습니다.
RAW_RUN_DIR = REPO_ROOT / 'backend/data/onbid/your-private-run'
OUTPUT_ROOT = REPO_ROOT / 'backend/data/onbid/notebook-results'

SCRIPTS = REPO_ROOT / 'backend/scripts'
required_scripts = ['onbid_data.py', 'onbid_model.py', 'onbid_explain.py']
if not all((SCRIPTS / name).is_file() for name in required_scripts):
    raise FileNotFoundError('REPO_ROOT를 backend/scripts가 있는 저장소로 지정하세요.')
if not (RAW_RUN_DIR / 'manifest.json').is_file():
    raise FileNotFoundError('RAW_RUN_DIR를 승인된 원본 페이지와 manifest.json이 있는 폴더로 지정하세요.')
missing = [name for name in ['numpy', 'pandas', 'sklearn', 'joblib']
           if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError('현재 커널에 backend/requirements-ml.txt를 설치한 뒤 다시 실행하세요: ' + ', '.join(missing))
sys.path.insert(0, str(SCRIPTS))
import pandas as pd
import numpy as np
import sklearn
from onbid_data import preprocess
from onbid_model import train, load_bundle, build_evidence, FEATURES
from onbid_explain import make_fact_cards, explain
print({'python': sys.version.split()[0], 'pandas': pd.__version__,
       'numpy': np.__version__, 'sklearn': sklearn.__version__})


## 원본 검증과 전처리

기존 수집기의 해시·완료 여부 검사를 거쳐 전처리합니다. 원본 페이지는 읽고, 같은 수집 폴더의 `processed/`에 정제 CSV·제외 사유·품질 보고서를 생성합니다. 기존 정제 결과가 있으면 다시 생성합니다.

폐차·다수 차량 묶음·점검용 등 단일 차량 학습에 맞지 않는 자료를 격리합니다. 가격 크기만으로 자료를 삭제하지 않습니다. 실제 규칙과 버전은 `onbid_data.py`를 기준으로 합니다.


In [ ]:
quality = preprocess(RAW_RUN_DIR)
DATA_CSV = RAW_RUN_DIR / 'processed/passenger_clean.csv'
print(pd.DataFrame([{
    '원본 행': quality['raw_rows'],
    '승용 범주 행': quality['passenger_category_rows'],
    '학습용 정제 행': quality['clean_passenger_rows'],
    '반복 차량 그룹': quality['repeated_vehicle_groups'],
}]).to_string(index=False))
print(pd.DataFrame(quality['missing_among_passenger']).T.to_string())


## 기준선과 비부스팅 모델 비교

시간과 차량 그룹을 분리하고, 전처리는 훈련 구간에만 적합합니다. 전체 중앙값·조건별 중앙값·Extra Trees·Random Forest를 비교해 검증 MAE로 선택합니다. 별도 잔차 보정과 최종 시험을 수행하며 새 결과를 운영 모델로 자동 승격하지 않습니다.

전체 시험 성능과 사전 지원 규칙의 제공률·조건부 성능을 함께 읽으세요. 조건부 성능은 전체 결과를 대신하지 않습니다. 자료가 부족하면 모델 함수가 명시적으로 실패합니다.


In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
MODEL_DIR = OUTPUT_ROOT / run_id
metrics = train(DATA_CSV, MODEL_DIR, n_estimators=200, n_jobs=2)
bundle = load_bundle(MODEL_DIR / 'model_bundle.joblib')  # 방금 직접 만든 파일만 읽습니다.
print('선택 모델:', metrics['selected_model'])
print('검증 구간 비교:')
print(pd.DataFrame(metrics['validation_scores']).T.to_string())
print('전체 최종 시험:')
print(pd.DataFrame([metrics['test_metrics']]).to_string(index=False))
support = metrics['service_support_audit']
print('제공 범위:', {name: support[name] for name in
                     ['test_rows', 'supported_rows', 'deferred_rows', 'supported_percent']})
print('제공 대상 조건부 성능:', support['supported_test_metrics'])
print('시간·그룹 분리:', json.dumps(metrics['split'], ensure_ascii=False, indent=2))


In [ ]:
print('검증 자료의 permutation importance:')
print(pd.DataFrame(metrics['validation_feature_importance']).to_string(index=False))
print('중요도는 모델의 정보 의존도이며 개별 차량 감가의 원인 증명이 아닙니다.')
subgroups = json.loads((MODEL_DIR / 'subgroup_metrics.json').read_text(encoding='utf-8'))
print(pd.DataFrame(subgroups).to_string(index=False))


## 시나리오와 검증된 기본 설명

아래 예시는 새로 학습한 자료의 대표 조건을 사용하는 동작 시연입니다. 사용자의 실제 차량이나 독립 성능 시험이 아닙니다. 모델/제조사 조합 또는 수치가 지원 범위를 벗어나면 가격 제시를 유보합니다.

현재·6·12개월 차령과 주행거리만 바꿉니다. 미래 시장가격·실제 매입가·최적 매도일을 보장하지 않습니다. 설명은 기본 템플릿으로만 만들며 LLM이나 외부 API를 호출하지 않습니다.


In [ ]:
reference = bundle['reference_train']
largest_group = reference.groupby(['model', 'make']).size().idxmax()
examples = reference[(reference['model'] == largest_group[0]) &
                     (reference['make'] == largest_group[1])]
example = examples.sort_values('odometer_km').iloc[len(examples) // 2]
vehicle = {key: None if pd.isna(value) else value
           for key, value in example[FEATURES].to_dict().items()}
# 사용자 입력을 적용하려면 vehicle 값을 수정하세요.
evidence = build_evidence(bundle, vehicle, monthly_km=1000,
                          months=(0, 6, 12), monthly_upkeep_krw=150000)
print('동작 시연 입력:', json.dumps(vehicle, ensure_ascii=False))
print('지원 상태:', evidence['status'])
print(pd.DataFrame(evidence['scenarios']).to_string(index=False))
print('유보 이유:', evidence['defer_reasons'])
print('경고:', evidence['warnings'])
explanation = explain(make_fact_cards(evidence))
print('설명 모드:', explanation['mode'])
print(explanation['text'])


In [ ]:
(MODEL_DIR / 'evidence.json').write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2), encoding='utf-8')
(MODEL_DIR / 'explanation.json').write_text(
    json.dumps(explanation, ensure_ascii=False, indent=2), encoding='utf-8')
print('로컬 결과 폴더:', MODEL_DIR)
print('이 결과·모델·원본·노트북 실행 출력은 Git 커밋 대상이 아닙니다.')


기술 근거와 검증 범위: [데이터](../docs/ONBID-DATA.md), [모델](../docs/ONBID-MODEL.md), [인수 메모](../docs/DELIVERY-NOTES.md).

이 파일의 준비 절차와 데이터가 내장된 별도 Colab 버전의 차이는 [notebooks README](README.md)를 확인하세요. 이 노트북 실행은 실제 Colab·AWS 운영·소비자 시장 성능 검증을 자동으로 의미하지 않습니다.
